# 这是整个项目核心的技术重点。
# 目标：在时间 t 计算的特征，只能用 t 及之前的数据；
# 标签用 t+1 的数据。

## 把清洗后的价格数据（prices.parquet）加工成带技术指标 + 预测标签的数据集（features.parquet），供后续机器学习使用。
## 1. 写 src/features/build_features.py
- 详见 /home/mlops/quant-mlops-project/src/features/build_features.py

In [ ]:
# 执行特征构建脚本，生成完整数据集
# 终端执行记录：
(quant-mlops-project) mlops@MS-CEVXSRKPSHOI:~/quant-mlops-project$ uv run python src/features/build_features.py
dropped 210 rows (warmup + label NaN), kept 29310
saved -> data/features/features.parquet
date range: 2015-02-02 ~ 2026-09-28
features: ['ret_1d', 'ret_5d', 'ret_20d', 'vol_5d', 'vol_20d', 'vol_ratio', 'price_pos', 'ma_ratio']

In [1]:
import pandas  as pd
df = pd.read_parquet('/home/mlops/quant-mlops-project/data/features/features.parquet')

In [2]:
df.describe()

,date,open,high,low,close,volume,ret_1d,ret_5d,ret_20d,vol_5d,vol_20d,vol_ma20,vol_ratio,high_20,low_20,price_pos,ma_5,ma_20,ma_ratio,label
count,291878,291878.000000,291878.000000,291878.000000,291878.000000,2.918780e+05,291878.000000,291878.000000,291878.000000,291878.000000,291878.000000,2.918780e+05,291878.000000,291878.000000,291878.000000,2.918780e+05,291878.000000,291878.000000,291878.000000,291878.000000
mean,2020-12-02 11:09:40.656000,153.050579,154.708436,151.353347,153.067542,1.730251e+07,0.000727,0.003568,0.014203,0.015351,0.016582,1.732962e+07,1.008047,161.010972,143.176801,5.670061e-01,152.905459,152.301048,1.004266,0.000723
min,2015-02-02 00:00:00,0.462494,0.471493,0.453397,0.466182,1.210000e+04,-0.440438,-0.489181,-0.712422,0.000227,0.002383,2.885000e+05,0.011747,0.490324,0.453397,-3.422446e-15,0.466994,0.474475,0.500001,-0.440438
25%,2018-01-05 00:00:00,51.883941,52.404952,51.317353,51.896894,2.532042e+06,-0.007686,-0.016501,-0.029872,0.008161,0.010442,2.743711e+06,0.755934,54.507804,48.756164,3.042249e-01,51.867630,51.729507,0.986752,-0.007688
50%,2020-12-03 00:00:00,98.663811,99.602077,97.644732,98.643845,5.428250e+06,0.000754,0.003827,0.013061,0.012350,0.014194,5.817280e+06,0.919640,103.507066,92.550003,6.073410e-01,98.581773,98.220804,1.005302,0.000748
75%,2023-10-30 00:00:00,194.248056,196.359454,192.159148,194.289249,1.294038e+07,0.009245,0.023667,0.055622,0.018775,0.019667,1.316343e+07,1.144961,204.449997,181.546378,8.430409e-01,194.112739,193.264890,1.022712,0.009236
max,2026-09-28 00:00:00,1274.130005,1292.650024,1243.229980,1280.339966,3.692928e+09,0.612184,0.847815,1.147065,0.274263,0.139917,1.127078e+09,11.432927,1292.650024,1147.000000,1.000000e+00,1252.146021,1205.916315,1.429156,0.612184
std,NaN,156.836279,158.604065,154.987665,156.829451,5.488572e+07,0.019160,0.041518,0.081502,0.011740,0.009729,5.065193e+07,0.426757,165.422353,146.406415,3.010050e-01,156.622837,155.901975,0.035003,0.019158


In [3]:
df.head()

,date,ticker,open,high,low,close,volume,ret_1d,ret_5d,ret_20d,...,vol_20d,vol_ma20,vol_ratio,high_20,low_20,price_pos,ma_5,ma_20,ma_ratio,label
0,2015-02-02,AAPL,26.099665,26.347284,25.664117,26.227896,250956400,0.012547,0.048895,0.085064,...,0.023813,262959580.0,0.954354,26.530784,23.132638,0.910867,25.608398,24.564523,1.042495,0.000168
1,2015-02-03,AAPL,26.199147,26.329590,26.002378,26.232311,207662800,0.000168,0.087136,0.116706,...,0.022587,260485620.0,0.797214,26.530784,23.132638,0.912166,26.028910,24.701599,1.053734,0.007670
2,2015-02-04,AAPL,26.199149,26.643540,26.157142,26.433504,280598800,0.007670,0.036857,0.125164,...,0.022551,261356140.0,1.073626,26.643540,23.258654,0.937949,26.216836,24.848623,1.055062,0.007138
3,2015-02-05,AAPL,26.639937,26.686550,26.469026,26.622181,168984800,0.007138,0.012728,0.117526,...,0.022477,261784200.0,0.645512,26.686550,23.258654,0.981222,26.283756,24.988610,1.051829,-0.008421
4,2015-02-06,AAPL,26.639926,26.690978,26.291445,26.397987,174826400,-0.008421,0.019114,0.067113,...,0.021310,258652620.0,0.675912,26.690978,23.258654,0.914638,26.382776,25.071622,1.052296,0.006643


In [4]:
df.tail()

,date,ticker,open,high,low,close,volume,ret_1d,ret_5d,ret_20d,...,vol_20d,vol_ma20,vol_ratio,high_20,low_20,price_pos,ma_5,ma_20,ma_ratio,label
291873,2026-09-22,ZTS,71.980003,73.459999,71.239998,72.860001,3840400,0.021450,-0.007762,-0.055851,...,0.014740,4730015.0,0.811921,78.75,70.260002,0.306243,72.322000,74.559999,0.969984,-0.017156
291874,2026-09-23,ZTS,72.750000,73.550003,71.550003,71.610001,3828600,-0.017156,-0.019041,-0.073489,...,0.015042,4747655.0,0.806419,78.75,70.260002,0.159010,72.044000,74.275999,0.969950,-0.018713
291875,2026-09-24,ZTS,71.849998,72.379997,69.930000,70.269997,3369400,-0.018713,-0.037661,-0.093524,...,0.015315,4673785.0,0.720915,78.00,69.930000,0.042131,71.494000,73.913499,0.967266,0.011100
291876,2026-09-25,ZTS,70.000000,71.080002,69.519997,71.050003,4165600,0.011100,-0.004902,-0.052919,...,0.014254,4683435.0,0.889433,78.00,69.519997,0.180425,71.424001,73.714999,0.968921,0.005348
291877,2026-09-28,ZTS,70.500000,71.680000,70.080002,71.430000,3907400,0.005348,0.001402,-0.076416,...,0.012066,4636590.0,0.842731,78.00,69.519997,0.225236,71.444000,73.419500,0.973093,-0.015820


In [ ]:
# 抽查 AAPL 的前 25 行特征数据，目的是验证特征计算是否正确、有没有未来函数。
# 终端执行记录如下：
(quant-mlops-project) mlops@MS-CEVXSRKPSHOI:~/quant-mlops-project$ uv run python -c "
import pandas as pd
df = pd.read_parquet('data/features/features.parquet')
a = df[df['ticker']=='AAPL'].sort_values('date').head(25).reset_index(drop=True)
pd.set_option('display.width', 200)
print(a[['date','close','ret_1d','vol_5d','label']].to_string())
"
         date      close    ret_1d    vol_5d     label
0  2015-02-02  26.227894  0.012547  0.036233  0.000169
1  2015-02-03  26.232317  0.000169  0.027687  0.007669
2  2015-02-04  26.433504  0.007669  0.016790  0.007137
3  2015-02-05  26.622171  0.007137  0.010585 -0.008421
4  2015-02-06  26.397997 -0.008421  0.008142  0.006642
5  2015-02-09  26.573341  0.006642  0.006892  0.019212
6  2015-02-10  27.083858  0.019212  0.009823  0.023439
7  2015-02-11  27.718668  0.023439  0.012484  0.012652
8  2015-02-12  28.069372  0.012652  0.012456  0.004903
9  2015-02-13  28.206989  0.004903  0.007953  0.005901
10 2015-02-17  28.373451  0.005901  0.008115  0.006963
11 2015-02-18  28.571011  0.006963  0.007692 -0.002098
12 2015-02-19  28.511076 -0.002098  0.005277  0.008174
13 2015-02-20  28.744129  0.008174  0.004027  0.027028
14 2015-02-23  29.521011  0.027028  0.010752 -0.006241
15 2015-02-24  29.336777 -0.006241  0.012848 -0.025573
16 2015-02-25  28.586548 -0.025573  0.019328  0.012656
17 2015-02-26  28.948339  0.012656  0.019993 -0.015028
18 2015-02-27  28.513294 -0.015028  0.021208  0.004904
19 2015-03-02  28.653130  0.004904  0.015260  0.002091
20 2015-03-03  28.713053  0.002091  0.015658 -0.006339
21 2015-03-04  28.531042 -0.006339  0.010654 -0.016570
22 2015-03-05  28.058275 -0.016570  0.009715  0.001503
23 2015-03-06  28.100443  0.001503  0.008716  0.004265
24 2015-03-09  28.220301  0.004265  0.008577 -0.020686

### 验证逻辑：
- 第 t 行的 ret_1d = (close[t] / close[t-1]) - 1
- 第 t 行的 label = (close[t+1] / close[t]) - 1
- label 和下一行的 ret_1d 应该相等（因为 label 是“明天的收益”，下一行的 ret_1d 就是“明天相对今天的收益”）
- 如果对得上，point-in-time 就是对的。